In [2]:
# Cell 1: imports & basic config

import pandas as pd
import numpy as np
from pathlib import Path

import torch
import lightning.pytorch as pl
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.metrics import MAE

print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch: 2.5.1+cu121
Lightning: 2.5.6
CUDA available: True


In [3]:
# Cell 2: load parquet file

DATA_PATH = Path(r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet")

df_raw = pd.read_parquet(DATA_PATH).copy()
print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()


Raw shape: (61473, 14)
Columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2016-01-07 00:00:00+00:00,542,1.07764,1.07832,1.07744,1.07778,1.07757,1.07823,1.07735,1.07770,1.07772,1.07840,1.07752,1.07787
1,2016-01-07 01:00:00+00:00,3167,1.07776,1.08100,1.07748,1.08029,1.07768,1.08092,1.07740,1.08020,1.07784,1.08109,1.07756,1.08038
2,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,1.08035,1.08184,1.08005,1.08159
3,2016-01-07 03:00:00+00:00,914,1.08156,1.08257,1.08150,1.08187,1.08147,1.08249,1.08142,1.08178,1.08164,1.08265,1.08157,1.08196
4,2016-01-07 04:00:00+00:00,649,1.08190,1.08256,1.08156,1.08236,1.08182,1.08247,1.08147,1.08228,1.08199,1.08264,1.08163,1.08245


In [4]:
# Cell 3: time handling, series_id, time_idx

TIME_COL = "time"
PRICE_COL = "mid_c"

df = df_raw.copy()

# ensure timezone-aware datetime
df[TIME_COL] = pd.to_datetime(df[TIME_COL], utc=True)
df = df.sort_values(TIME_COL).reset_index(drop=True)

# single series id
df["series_id"] = "eurusd"

# time index in hours from start
df["time_idx"] = (df[TIME_COL] - df[TIME_COL].min()).dt.total_seconds() // 3600
df["time_idx"] = df["time_idx"].astype(int)

# hour & day_of_week (we'll make them categoricals later)
df["hour"] = df[TIME_COL].dt.hour
df["day_of_week"] = df[TIME_COL].dt.dayofweek   # 0=Mon..6=Sun

print(df[[TIME_COL, "time_idx", "hour", "day_of_week"]].head())
print("Min time_idx:", df["time_idx"].min(), "Max time_idx:", df["time_idx"].max())


                       time  time_idx  hour  day_of_week
0 2016-01-07 00:00:00+00:00         0     0            3
1 2016-01-07 01:00:00+00:00         1     1            3
2 2016-01-07 02:00:00+00:00         2     2            3
3 2016-01-07 03:00:00+00:00         3     3            3
4 2016-01-07 04:00:00+00:00         4     4            3
Min time_idx: 0 Max time_idx: 86565


In [5]:
# Cell 4: add technical indicators

from ta.momentum import RSIIndicator
from ta.trend import MACD, EMAIndicator
from ta.volatility import BollingerBands, AverageTrueRange

df_feat = df.copy()

# RSI(14)
df_feat["rsi"] = RSIIndicator(
    close=df_feat["mid_c"],
    window=14
).rsi()

# MACD(12,26,9)
macd = MACD(
    close=df_feat["mid_c"],
    window_slow=26,
    window_fast=12,
    window_sign=9,
)
df_feat["macd"] = macd.macd()
df_feat["macd_signal"] = macd.macd_signal()

# Bollinger Bands (20, 2)
bb = BollingerBands(
    close=df_feat["mid_c"],
    window=20,
    window_dev=2,
)
df_feat["bb_upper"]  = bb.bollinger_hband()
df_feat["bb_middle"] = bb.bollinger_mavg()
df_feat["bb_lower"]  = bb.bollinger_lband()

# ATR(14) using mid prices
atr = AverageTrueRange(
    high=df_feat["mid_h"],
    low=df_feat["mid_l"],
    close=df_feat["mid_c"],
    window=14,
)
df_feat["atr14"] = atr.average_true_range()

# EMAs
df_feat["ema_5"]   = EMAIndicator(close=df_feat["mid_c"], window=5).ema_indicator()
df_feat["ema_20"]  = EMAIndicator(close=df_feat["mid_c"], window=20).ema_indicator()
df_feat["ema_50"]  = EMAIndicator(close=df_feat["mid_c"], window=50).ema_indicator()
df_feat["ema_200"] = EMAIndicator(close=df_feat["mid_c"], window=200).ema_indicator()

print("Shape with indicators:", df_feat.shape)
df_feat[["mid_c","rsi","macd","macd_signal","bb_upper","bb_lower","atr14","ema_20"]].head()


Shape with indicators: (61473, 29)


,mid_c,rsi,macd,macd_signal,bb_upper,bb_lower,atr14,ema_20
0,1.07778,NaN,NaN,NaN,NaN,NaN,0.0,NaN
1,1.08029,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2,1.08152,NaN,NaN,NaN,NaN,NaN,0.0,NaN
3,1.08187,NaN,NaN,NaN,NaN,NaN,0.0,NaN
4,1.08236,NaN,NaN,NaN,NaN,NaN,0.0,NaN


In [6]:
# Cell 5: target = 4h forward log-return

HORIZON = 4  # 4 hours ahead

df_feat["target_4h_return"] = np.log(
    df_feat["mid_c"].shift(-HORIZON) / df_feat["mid_c"]
)

# Drop rows where target or important indicators are NaN
needed_cols = [
    "target_4h_return",
    "rsi", "macd", "macd_signal",
    "bb_upper", "bb_middle", "bb_lower",
    "atr14",
    "ema_5", "ema_20", "ema_50", "ema_200",
]

df_model = df_feat.dropna(subset=needed_cols).reset_index(drop=True)

print("After dropna, shape:", df_model.shape)
df_model[[TIME_COL, "mid_c", "target_4h_return"]].head()


After dropna, shape: (61270, 30)


,time,mid_c,target_4h_return
0,2016-01-19 07:00:00+00:00,1.08652,-0.000203
1,2016-01-19 08:00:00+00:00,1.08846,-0.000919
2,2016-01-19 09:00:00+00:00,1.08724,-0.000570
3,2016-01-19 10:00:00+00:00,1.08730,0.001682
4,2016-01-19 11:00:00+00:00,1.08630,0.003208


In [7]:
# Cell 6: make hour/day_of_week categorical (string) for TFT

df_model["hour"] = df_model["hour"].astype(str)
df_model["day_of_week"] = df_model["day_of_week"].astype(str)

print(df_model[["time", "hour", "day_of_week"]].head())
print(df_model[["hour", "day_of_week"]].dtypes)

print("Total rows:", len(df_model))
print("Min time_idx:", df_model["time_idx"].min(), "Max time_idx:", df_model["time_idx"].max())


                       time hour day_of_week
0 2016-01-19 07:00:00+00:00    7           1
1 2016-01-19 08:00:00+00:00    8           1
2 2016-01-19 09:00:00+00:00    9           1
3 2016-01-19 10:00:00+00:00   10           1
4 2016-01-19 11:00:00+00:00   11           1
hour           object
day_of_week    object
dtype: object
Total rows: 61270
Min time_idx: 295 Max time_idx: 86561


In [8]:
# Cell 7: time-based split

max_time = df_model["time_idx"].max()
min_time = df_model["time_idx"].min()
total_span = max_time - min_time

train_cutoff = min_time + int(total_span * 0.8)
val_cutoff   = min_time + int(total_span * 0.9)

train_df = df_model[df_model["time_idx"] <= train_cutoff].copy()
val_df   = df_model[(df_model["time_idx"] > train_cutoff) & (df_model["time_idx"] <= val_cutoff)].copy()
test_df  = df_model[df_model["time_idx"] > val_cutoff].copy()

print("Train:", train_df.shape)
print("Val:",   val_df.shape)
print("Test:",  test_df.shape)

print("Train time range:", train_df["time"].min(), "->", train_df["time"].max())
print("Val   time range:", val_df["time"].min(),   "->", val_df["time"].max())
print("Test  time range:", test_df["time"].min(),  "->", test_df["time"].max())


Train: (49012, 30)
Val: (6127, 30)
Test: (6131, 30)
Train time range: 2016-01-19 07:00:00+00:00 -> 2023-12-01 21:00:00+00:00
Val   time range: 2023-12-03 22:00:00+00:00 -> 2024-11-27 06:00:00+00:00
Test  time range: 2024-11-27 07:00:00+00:00 -> 2025-11-21 17:00:00+00:00


In [9]:
# Cell 8: TimeSeriesDataSet definitions

FEATURE_COLS_BASE = ["mid_o", "mid_h", "mid_l", "mid_c", "volume"]

FEATURE_COLS_IND = [
    "rsi",
    "macd",
    "macd_signal",
    "bb_lower",
    "bb_middle",
    "bb_upper",
    "atr14",
    "ema_5",
    "ema_20",
    "ema_50",
    "ema_200",
]

FEATURE_COLS_EXT = FEATURE_COLS_BASE + FEATURE_COLS_IND

MAX_ENCODER_LENGTH = 96   # 96 hours history (~4 days)
MAX_PRED_LENGTH    = 1    # predict a single 4h return

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target_4h_return",
    group_ids=["series_id"],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PRED_LENGTH,

    time_varying_unknown_reals=FEATURE_COLS_EXT,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],

    categorical_encoders={
        "hour": NaNLabelEncoder(add_nan=False),
        "day_of_week": NaNLabelEncoder(add_nan=False),
        "series_id": NaNLabelEncoder(add_nan=False),
    },

    target_normalizer=None,       # keep raw log-return for easy integration
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    data=val_df,
    stop_randomization=True,
)

test = TimeSeriesDataSet.from_dataset(
    training,
    data=test_df,
    stop_randomization=True,
)

print("Number of samples - train:", len(training))
print("Number of samples - val:",   len(validation))
print("Number of samples - test:",  len(test))


Number of samples - train: 29118
Number of samples - val: 3572
Number of samples - test: 3668


In [10]:
# Cell 9: DataLoaders

BATCH_SIZE = 256  # your 4070 can handle this easily

train_loader = training.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

test_loader = test.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print("Train batches:", len(train_loader))
print("Val batches:",   len(val_loader))
print("Test batches:",  len(test_loader))


Train batches: 113
Val batches: 14
Test batches: 15


In [11]:
# Cell 10: create TFT model

import torch

torch.set_float32_matmul_precision("medium")
pl.seed_everything(42)

LEARNING_RATE = 1e-3
HIDDEN_SIZE = 32
ATTN_HEADS = 4
DROPOUT = 0.1
HIDDEN_CONT = 16

tft_4h = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTN_HEADS,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONT,
    loss=MAE(),                # MAE on log-return
    output_size=1,             # 1-step forecast
    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=4,
)

print("Number of parameters (k):", tft_4h.size() / 1e3)


Seed set to 42


Number of parameters (k): 95.764


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [13]:
# UPDATED Cell 11: Trainer & training (no half precision)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="eurusd_h1_tft_4h-{epoch:02d}-{val_loss:.6f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    min_delta=1e-5,
)

logger = CSVLogger(
    save_dir="logs",
    name="eurusd_h1_tft_4h",
)

trainer = Trainer(
    max_epochs=60,
    accelerator=accelerator,
    devices=devices,
    gradient_clip_val=0.5,
    # 👇 IMPORTANT: no AMP, stay in full float32
    precision=32,
    callbacks=[checkpoint_cb, earlystop_cb],
    logger=logger,
    log_every_n_steps=50,
    num_sanity_val_steps=0,
)

trainer.fit(
    model=tft_4h,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("Best checkpoint path:", checkpoint_cb.best_model_path)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 241    | train
3  | prescalers                         | ModuleDict                      | 576    | train
4  | static_variable_selection          | VariableSelectionNetwork        | 2.0 K  | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 38.5 K | train
6  | decoder_variable_selection         | VariableSelectionNetwork        | 2.3 K  | train
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K  | train
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 4.3 K  

Epoch 24: 100%|██████████| 113/113 [00:48<00:00,  2.34it/s, v_num=1, train_loss_step=0.00191, val_loss=0.002, train_loss_epoch=0.00181]  
Best checkpoint path: logs\eurusd_h1_tft_4h\version_1\checkpoints\eurusd_h1_tft_4h-epoch=14-val_loss=0.001203.ckpt


In [14]:
# Cell 12: evaluation on test set

from math import copysign

# load best model
best_tft_4h = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint_cb.best_model_path
)

best_tft_4h.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
best_tft_4h.to(device)

all_preds = []
all_targets = []

with torch.no_grad():
    for x, y in test_loader:
        # y is a tuple: (target, weight) usually
        y_t = y[0]                      # [B, max_pred_len]
        x = {k: v.to(device) for k, v in x.items()}

        y_hat = best_tft_4h(x)["prediction"]   # [B, max_pred_len]
        all_preds.append(y_hat.detach().cpu().reshape(-1))
        all_targets.append(y_t.detach().cpu().reshape(-1))

preds = torch.cat(all_preds)
targets = torch.cat(all_targets)

print("Preds shape:", preds.shape)
print("Targets shape:", targets.shape)

# MAE in log-return
mae_ret = torch.mean(torch.abs(preds - targets)).item()

# Direction accuracy on 4h move
sign_pred = torch.sign(preds)
sign_true = torch.sign(targets)

# treat zeros as no-move; we'll count only where target != 0
mask = sign_true != 0
dir_acc = (sign_pred[mask] == sign_true[mask]).float().mean().item()

print(f"Test MAE (4h log-return units): {mae_ret:.8f}")
print(f"Direction accuracy (4h horizon): {dir_acc:.4f} ({dir_acc*100:.2f}%)")


Preds shape: torch.Size([3668])
Targets shape: torch.Size([3668])
Test MAE (4h log-return units): 0.00161353
Direction accuracy (4h horizon): 0.4820 (48.20%)


In [15]:
# Cell 13: convert MAE in return → price → pips (approx)

# mean price on test range
val_mask = df_model["time_idx"] > val_cutoff
mean_price_test = df_model.loc[val_mask, "mid_c"].mean()

mae_price = mae_ret * mean_price_test
pip = 0.0001
mae_pips = mae_price / pip

print(f"Mean test price: {mean_price_test:.5f}")
print(f"Approx. MAE in price: {mae_price:.6f}")
print(f"Approx. MAE in pips (4h): {mae_pips:.3f}")


Mean test price: 1.11879
Approx. MAE in price: 0.001805
Approx. MAE in pips (4h): 18.052
